# Notebook 07 — Reciprocidade do Dígrafo Original (Item 4 da Parte III)

**Aluno:** Vitor Renato Santos dos Santos | **Matrícula:** 222217109 — UFBA
**Dataset:** ego-Gplus (G2) — SNAP | **Grafo:** G2 ($n = 107.614$, $m_{arcos} = 13.673.453$)

Este notebook apresenta a análise de reciprocidade do dígrafo original,
incluindo métricas globais, validação interna, comparação com baseline
aleatório, distribuições de grau dirigido, reciprocidade por faixa de grau
e relação com a maior SCC.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RESULTS_DIR = "results"
FIGURES_DIR = "figures"

## 1. Resultados Pré-Computados
Os resultados foram gerados pelo script `scripts/run_parte3_extra.py`.

In [ ]:
with open(os.path.join(RESULTS_DIR, "parte3_reciprocidade.json"), "r", encoding="utf-8") as f:
    res = json.load(f)

print(json.dumps(res, indent=2, ensure_ascii=False))

## 2. Reciprocidade Global

In [ ]:
rg = res["reciprocidade_global"]
print(f"Pares recíprocos: {rg['pares_reciprocos']:,}")
print(f"Fração de arcos em laços bidirecionais: {rg['fracao_pct']:.2f}%")
print(f"Validação interna: arcos − pares = {res['validacao_interna']['arcos_unicos_menos_pares']:,}")
print(f"  Esperado: {res['validacao_interna']['m_undirected_esperado']:,}")
print(f"  Status: {'PASSOU ✓' if res['validacao_interna']['fechou'] else 'FALHOU ✗'}")

## 3. Baseline (Dígrafo Aleatório)

In [ ]:
rb = res["baseline_aleatorio"]
print(f"Densidade do dígrafo: ρ = {rb['densidade_digrafo']:.8f}")
print(f"Reciprocidade esperada em dígrafo aleatório: {rb['reciprocidade_esperada_aleatoria']:.8f}")
print(f"Razão observado/esperado: {rb['razao_observado_esperado']:.2f}×")
print()
print("Interpretação: a reciprocidade observada (≈21%) é cerca de 100× maior")
print("que a esperada em um dígrafo aleatório com mesma densidade (~0.21%),")
print("indicando forte tendência à mutualidade nas conexões do Google+.")

## 4. Distribuições de Grau Dirigido (In-degree e Out-degree)

In [ ]:
fig_path = os.path.join(FIGURES_DIR, "parte3_reciprocidade_graus_dirigidos.png")
if os.path.exists(fig_path):
    from IPython.display import Image, display
    display(Image(filename=fig_path))
else:
    print(f"Figura não encontrada: {fig_path}")

In [ ]:
gd = res["graus_dirigidos"]
print(f"In-degree médio: {gd['in_degree_medio']:.2f}")
print(f"Out-degree médio: {gd['out_degree_medio']:.2f}")
print(f"In-degree máximo: {gd['in_degree_max']:,}")
print(f"Out-degree máximo: {gd['out_degree_max']:,}")
print(f"Correlação in vs out: r = {gd['correlacao_in_out']:.4f}")

## 5. Reciprocidade por Nó

In [ ]:
rn = res["reciprocidade_por_no"]
print(f"Nós com reciprocidade total: {rn['nos_reciprocidade_total']:,} ({rn['frac_reciprocidade_total_pct']:.2f}%)")
print(f"Nós sem nenhum arco recíproco: {rn['nos_sem_reciprocidade']:,} ({rn['frac_sem_reciprocidade_pct']:.2f}%)")

## 6. Reciprocidade por Faixa de Grau

In [ ]:
df_bands = pd.read_csv(os.path.join(RESULTS_DIR, "parte3_reciprocidade_por_faixa.csv"))
print(df_bands.to_string(index=False))

In [ ]:
fig_path2 = os.path.join(FIGURES_DIR, "parte3_reciprocidade_por_faixa.png")
if os.path.exists(fig_path2):
    from IPython.display import Image, display
    display(Image(filename=fig_path2))

**Observação:** Nós de baixo grau tendem a ter reciprocidade mais alta em fração,
enquanto hubs acumulam muitos arcos unilaterais (seguidores que não são seguidos de volta).
Isso é consistente com a dinâmica de redes sociais onde celebridades recebem muitos seguidores
mas seguem poucas pessoas de volta.

## 7. Relação com a SCC

In [ ]:
scc = res["scc"]
print(f"Total de SCCs: {scc['total_sccs']:,}")
print(f"Maior SCC: {scc['maior_scc_nos']:,} nós ({scc['maior_scc_frac_pct']:.2f}%)")
print(f"Reciprocidade na maior SCC: {scc['frac_reciprocidade_na_scc_pct']:.2f}% dos arcos recíprocos de saída")
print()
print("A maior SCC concentra a vasta maioria da reciprocidade.")
print("Isto é natural: a SCC exige conectividade bidirecional entre seus membros,")
print("logo pares recíprocos tendem a residir dentro dela.")

## 8. Conclusões

1. **Reciprocidade forte:** ~21% dos arcos são bidirecionais, ~100× acima do esperado
   em um dígrafo aleatório com mesma densidade.
2. **Validação fechou:** 13.673.453 − 1.435.168 = 12.238.285 ✓
3. **Nós periféricos mais recíprocos:** fração recíproca inversamente correlacionada
   com o grau total — consistente com "círculos sociais" íntimos (hipótese).
4. **SCC concentra reciprocidade:** >90% dos arcos recíprocos estão na maior SCC (69.501 nós).
5. **Correlação in-out positiva:** indica que quem segue muitos também tende a ser seguido.